In [ ]:
import json
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.patches import FancyArrowPatch
import matplotlib.colors as mcolors

# Load the JSON data
with open('tg_data/graph.json', 'r') as f:
    data = json.load(f)

# Create a directed graph
G = nx.DiGraph()

# Dictionary to map state node IDs to their page node IDs
state_to_page = {}

# Process each page node and its state nodes
for page_node in data['page_nodes']:
    page_id = page_node['id']
    page_name = page_node['name']
    
    # Add page node
    page_node_id = f"P{page_id}"
    G.add_node(page_node_id, type='page', name=page_name)
    
    # Add state nodes and connect them to their parent page node
    for state_node in page_node['state_nodes']:
        state_id = state_node['id']
        state_node_id = f"S{state_id}"
        
        G.add_node(state_node_id, type='state')
        G.add_edge(page_node_id, state_node_id, style='dashed', color='gray')
        
        # Map state ID to page ID
        state_to_page[state_id] = page_id
        
        # Process transitions between state nodes
        for transition in state_node['transitions']:
            source = f"S{state_node['id']}"
            target = f"S{transition['target_state_node_id']}"
            event = transition['event']
            
            # Extract event info for edge label
            if 'event_type' in event:
                event_type = event['event_type']
                event_id = event['event_id']
                edge_label = f"{event_type} ({event_id})"
                
                # Add edge with attributes
                G.add_edge(source, target, label=edge_label, event_type=event_type, style='solid', color='black')

for internal_state_node in data['internal_state_nodes']:
    internal_state_id = internal_state_node['id']
    internal_state_node_id = f"S{internal_state_id}"
    
    # Add internal state node
    G.add_node(internal_state_node_id, type='internal_state')
    
    # Process transitions between internal state nodes
    for transition in internal_state_node['transitions']:
        source = f"S{internal_state_node['id']}"
        target = f"S{transition['target_state_node_id']}"
        event = transition['event']
        
        # Extract event info for edge label
        if 'event_type' in event:
            event_type = event['event_type']
            event_id = event['event_id']
            edge_label = f"{event_type} ({event_id})"
            
            # Add edge with attributes
            G.add_edge(source, target, label=edge_label, event_type=event_type, style='solid', color='black')

# Set up the plot
plt.figure(figsize=(20, 15))

# Use a more hierarchical layout
pos = nx.kamada_kawai_layout(G)

# Get node types
node_types = nx.get_node_attributes(G, 'type')

# Draw nodes with different shapes and colors based on type
page_nodes = [node for node, type_val in node_types.items() if type_val == 'page']
state_nodes = [node for node, type_val in node_types.items() if type_val == 'state']
internal_state_nodes = [node for node, type_val in node_types.items() if type_val == 'internal_state']

# Create labels
labels = {}
for node in G.nodes():
    if node.startswith('P'):  # Page nodes
        page_id = int(node[1:])
        for page_node in data['page_nodes']:
            if page_node['id'] == page_id:
                labels[node] = f"Page {page_id}\n({page_node['name']})"
                break
    else:  # State nodes
        state_id = int(node[1:])
        screenshot = 'INTERNAL'
        for page_node in data['page_nodes']:
            for state_node in page_node['state_nodes']:
                if state_node['id'] == state_id:
                    screenshot = str(state_node['screenshot_path']).split('/')[-1]
                    break
        labels[node] = f"State {state_id}\nScreenshot: {screenshot}"

# Draw edges with different styles
edge_styles = nx.get_edge_attributes(G, 'style')
edge_colors = nx.get_edge_attributes(G, 'color')

# Draw dashed edges (page-to-state)
dashed_edges = [(u, v) for u, v, style in G.edges(data='style') if style == 'dashed']
nx.draw_networkx_edges(G, pos, edgelist=dashed_edges, style='dashed', edge_color='gray', width=0.5)

# Draw event edges with colors based on event type
event_edges = [(u, v) for u, v, style in G.edges(data='style') if style == 'solid']
edge_labels = {(u, v): G.edges[u, v]['label'] for u, v in event_edges if 'label' in G.edges[u, v]}

# Get unique event types for coloring
event_types = set()
for u, v, data in G.edges(data=True):
    if 'event_type' in data:
        event_types.add(data['event_type'])

# Create color map for event types
color_map = {event: color for event, color in zip(event_types, plt.cm.tab10.colors)}

# Find bidirectional edges
bidirectional_edges = []
for u, v in G.edges():
    if G.has_edge(v, u) and (v, u) not in bidirectional_edges:
        bidirectional_edges.append((u, v))
        bidirectional_edges.append((v, u))

# Draw edges grouped by event type, handling bidirectional edges separately
for event_type in event_types:
    # Get all edges of this type
    edges_of_type = [(u, v) for u, v, data in G.edges(data=True) 
                    if data.get('event_type') == event_type and data.get('style') == 'solid']
    
    # Shift bidirectional edges slightly to avoid overlap
    for u, v in edges_of_type:
        # Calculate the edge curvature based on whether it's bidirectional
        if (v, u) in G.edges() and u != v:
            # Draw curved edges for bidirectional connections
            nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=1.5, 
                                edge_color=color_map[event_type], arrows=True, 
                                arrowstyle='-|>', arrowsize=15,
                                connectionstyle='arc3,rad=0.2')  # Curve one direction
        else:
            # Draw straight edges for unidirectional connections
            nx.draw_networkx_edges(G, pos, edgelist=[(u, v)], width=1.5, 
                                edge_color=color_map[event_type], arrows=True, 
                                arrowstyle='-|>', arrowsize=15)

# Draw nodes last to make sure they're on top
nx.draw_networkx_nodes(G, pos, nodelist=page_nodes, node_color='lightgreen', 
                     node_shape='s', node_size=4000, alpha=0.8)
nx.draw_networkx_nodes(G, pos, nodelist=state_nodes, node_color='lightblue', 
                     node_shape='o', node_size=3000, alpha=0.8)
nx.draw_networkx_nodes(G, pos, nodelist=internal_state_nodes, node_color='lightyellow',
                        node_shape='o', node_size=3000, alpha=0.5)

# Draw the labels
nx.draw_networkx_labels(G, pos, labels, font_size=10, font_weight='bold')

# Draw edge labels slightly offset from the edges
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

# Create a legend for event types
legend_elements = [plt.Line2D([0], [0], color=color_map.get(event_type, 'black'), 
                            lw=2, label=f"{event_type}") for event_type in event_types]
legend_elements.append(plt.Line2D([0], [0], color='gray', linestyle='--', lw=1, label='contains'))
legend_elements.append(plt.Line2D([0], [0], marker='s', color='w', markerfacecolor='lightgreen', 
                                markersize=10, label='Page Node'))
legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='lightblue', 
                                markersize=10, label='State Node'))
legend_elements.append(plt.Line2D([0], [0], color='black', lw=2, label='Unidirectional'))
legend_elements.append(plt.Line2D([0], [0], color='black', lw=2, 
                                  marker='>', label='Bidirectional (two separate arrows)'))

plt.legend(handles=legend_elements, loc='upper right')

# Set title and layout
plt.title('App Hierarchical State Transition Graph', size=15)
plt.axis('off')
plt.tight_layout()
plt.show()